In [2]:
import os
import urllib.request

# Go up one level from /notebook into project root, then into /data
os.makedirs("../data", exist_ok=True)

url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
urllib.request.urlretrieve(url, "../data/results.csv")

print("Downloaded and saved!")

Downloaded and saved!


In [3]:
import pandas as pd
df = pd.read_csv("../data/results.csv")
print(df.shape)
df.head()

(49520, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [4]:
# Step 1: Drop rows where scores are missing (unplayed matches)
df = df.dropna(subset=['home_score', 'away_score']).copy()

print("Rows after dropping unplayed matches:", df.shape)

# Step 2: Define the H/D/A rule as a function
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'H'
    elif row['home_score'] < row['away_score']:
        return 'A'
    else:
        return 'D'

# Step 3: Apply it to every row to create our target column
df['result'] = df.apply(get_result, axis=1)

# Step 4: Sanity check
print(df['result'].value_counts())
df[['home_team', 'away_team', 'home_score', 'away_score', 'result']].head()

Rows after dropping unplayed matches: (49518, 9)
result
H    24264
A    13996
D    11258
Name: count, dtype: int64


,home_team,away_team,home_score,away_score,result
0,Scotland,England,0.0,0.0,D
1,England,Scotland,4.0,2.0,H
2,Scotland,England,2.0,1.0,H
3,England,Scotland,2.0,2.0,D
4,Scotland,England,3.0,0.0,H


In [5]:
# neutral=True means no home advantage, so we flip it with ~
df['is_home_advantage'] = (~df['neutral']).astype(int)

df[['home_team', 'away_team', 'neutral', 'is_home_advantage']].head()

,home_team,away_team,neutral,is_home_advantage
0,Scotland,England,False,1
1,England,Scotland,False,1
2,Scotland,England,False,1
3,England,Scotland,False,1
4,Scotland,England,False,1


In [6]:
# Build the "home team's perspective" table
home_view = df[['date', 'home_team', 'away_team', 'result']].copy()
home_view.rename(columns={'home_team': 'team', 'away_team': 'opponent'}, inplace=True)
home_view['points'] = home_view['result'].map({'H': 3, 'D': 1, 'A': 0})
home_view['was_home'] = True          # ← new line

# Build the "away team's perspective" table
away_view = df[['date', 'home_team', 'away_team', 'result']].copy()
away_view.rename(columns={'away_team': 'team', 'home_team': 'opponent'}, inplace=True)
away_view['points'] = away_view['result'].map({'H': 0, 'D': 1, 'A': 3})
away_view['was_home'] = False         # ← new line

# Stack both views into one team-centric table
team_matches = pd.concat([home_view, away_view], ignore_index=True)
team_matches = team_matches.sort_values(['team', 'date'])

team_matches.head()

,date,team,opponent,result,points,was_home
36263,2012-09-25,Abkhazia,Artsakh,D,1,True
85925,2012-10-21,Abkhazia,Artsakh,H,0,False
37320,2013-09-23,Abkhazia,South Ossetia,H,3,True
37794,2014-06-01,Abkhazia,Occitania,D,1,True
87317,2014-06-02,Abkhazia,Sápmi,A,3,False


In [7]:
team_matches[team_matches['team'] == 'Abkhazia'].head(6)

,date,team,opponent,result,points,was_home
36263,2012-09-25,Abkhazia,Artsakh,D,1,True
85925,2012-10-21,Abkhazia,Artsakh,H,0,False
37320,2013-09-23,Abkhazia,South Ossetia,H,3,True
37794,2014-06-01,Abkhazia,Occitania,D,1,True
87317,2014-06-02,Abkhazia,Sápmi,A,3,False
37826,2014-06-04,Abkhazia,South Ossetia,D,1,True


In [8]:
team_matches[(team_matches['date'] == '2014-06-02') & (team_matches['opponent'] == 'Abkhazia')]

,date,team,opponent,result,points,was_home
37799,2014-06-02,Sápmi,Abkhazia,A,0,True


In [9]:
team_matches['recent_form'] = (
    team_matches.groupby('team')['points']
    .apply(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

team_matches[team_matches['team'] == 'Brazil'].head(10)

,date,team,opponent,result,points,was_home,recent_form
49967,1914-09-20,Brazil,Argentina,H,0,False,NaN
49968,1914-09-27,Brazil,Argentina,A,3,False,0.000000
482,1916-07-08,Brazil,Chile,D,1,True,1.500000
50001,1916-07-10,Brazil,Argentina,D,1,False,1.333333
485,1916-07-12,Brazil,Uruguay,A,0,True,1.250000
50006,1916-07-18,Brazil,Uruguay,A,3,False,1.000000
50034,1917-10-03,Brazil,Argentina,H,0,False,1.600000
50038,1917-10-07,Brazil,Uruguay,H,0,False,1.000000
521,1917-10-12,Brazil,Chile,H,3,True,0.800000
50042,1917-10-16,Brazil,Uruguay,H,0,False,1.200000


In [10]:
# Safety net: unique ID for every match, so no merge can ever create duplicates
df['match_id'] = range(len(df))

# Merge home team's form (matched by date + home_team + away_team)
df = df.merge(
    team_matches[['date', 'team', 'opponent', 'recent_form']],
    left_on=['date', 'home_team', 'away_team'],
    right_on=['date', 'team', 'opponent'],
    how='left'
).rename(columns={'recent_form': 'home_team_form'}).drop(columns=['team', 'opponent'])

# Merge away team's form (matched by date + away_team + home_team)
df = df.merge(
    team_matches[['date', 'team', 'opponent', 'recent_form']],
    left_on=['date', 'away_team', 'home_team'],
    right_on=['date', 'team', 'opponent'],
    how='left'
).rename(columns={'recent_form': 'away_team_form'}).drop(columns=['team', 'opponent'])

# Collapse back to one row per real match, just in case
df = df.drop_duplicates(subset='match_id', keep='first').drop(columns='match_id')

print(df.shape)
df[['date', 'home_team', 'away_team', 'home_team_form', 'away_team_form']].tail()

(49518, 13)


,date,home_team,away_team,home_team_form,away_team_form
49609,2026-07-10,Spain,Belgium,2.6,2.2
49610,2026-07-11,Norway,England,2.4,2.6
49611,2026-07-11,Argentina,Switzerland,3.0,2.2
49612,2026-07-14,France,Spain,3.0,3.0
49613,2026-07-15,England,Argentina,2.6,3.0


In [11]:
print(df.columns.tolist())

['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'result', 'is_home_advantage', 'home_team_form', 'away_team_form']


In [12]:
from sklearn.model_selection import train_test_split

feature_cols = ['is_home_advantage', 'home_team_form', 'away_team_form']
target_col = 'result'

# Drop rows where any feature (or the target) is missing
# (early matches have NaN form, since there's no prior history yet)
model_df = df.dropna(subset=feature_cols + [target_col])

X = model_df[feature_cols]   # inputs the model learns from
y = model_df[target_col]     # what the model tries to predict

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Total usable rows:", model_df.shape[0])
print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

Total usable rows: 49215
Training rows: 39372
Test rows: 9843


In [13]:
from sklearn.linear_model import LogisticRegression

# max_iter: how many attempts the model gets to fine-tune itself.
# Default (100) sometimes isn't enough — 1000 is a safe, common bump.
model = LogisticRegression(max_iter=1000)

# Learn patterns from the training data
model.fit(X_train, y_train)

print("Model trained!")

Model trained!


In [14]:
predictions = model.predict(X_test[:5])
print("Predicted:", list(predictions))
print("Actual:   ", list(y_test[:5]))

Predicted: ['H', 'A', 'H', 'H', 'H']
Actual:    ['A', 'D', 'H', 'D', 'A']


In [15]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)   # predict on ALL 9,843 test matches, not just 5
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy: {acc:.2%}")

Accuracy: 51.69%


In [16]:
import os
import urllib.request

os.makedirs("../data", exist_ok=True)

url = "https://raw.githubusercontent.com/cnc8/fifa-world-ranking/master/fifa_ranking-2020-12-10.csv"
urllib.request.urlretrieve(url, "../data/fifa_ranking.csv")

print("FIFA ranking data downloaded and saved!")

FIFA ranking data downloaded and saved!


In [17]:
import numpy as np

# Load FIFA rankings and keep only the most recent snapshot
rank_df = pd.read_csv("../data/fifa_ranking.csv")
latest_date = rank_df['rank_date'].max()
latest_ranks = rank_df[rank_df['rank_date'] == latest_date][['country_full', 'rank']]
rank_lookup = dict(zip(latest_ranks['country_full'], latest_ranks['rank']))

# Manual fixes for teams whose names differ between the two datasets
name_fixes = {
    'South Korea': 'Korea Republic',
    'United States': 'USA',
    'Ivory Coast': "Côte d'Ivoire",
    'Iran': 'IR Iran',
    'Cape Verde': 'Cabo Verde',
    'DR Congo': 'Congo DR',
}

def get_rank(team):
    if pd.isna(team):
        return np.nan
    team = name_fixes.get(team, team)   # apply fix if this team needs one
    return rank_lookup.get(team, np.nan)  # np.nan if still not found

df['home_rank'] = df['home_team'].apply(get_rank)
df['away_rank'] = df['away_team'].apply(get_rank)
df['rank_difference'] = df['away_rank'] - df['home_rank']

print("Missing rank_difference:", df['rank_difference'].isna().sum(), "out of", df.shape[0])

Missing rank_difference: 6526 out of 49518


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

feature_cols = ['is_home_advantage', 'home_team_form', 'away_team_form', 'rank_difference']
target_col = 'result'

model_df = df.dropna(subset=feature_cols + [target_col])
X = model_df[feature_cols]
y = model_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Usable rows:", model_df.shape[0])
print(f"Accuracy with rank_difference: {acc:.2%}")

Usable rows: 42832
Accuracy with rank_difference: 56.45%


In [19]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print(f"Random Forest Accuracy: {rf_acc:.2%}")
print(f"Logistic Regression Accuracy: {acc:.2%}")   # from your last cell, for comparison

Random Forest Accuracy: 48.00%
Logistic Regression Accuracy: 56.45%


In [20]:
for depth in [3, 5, 8, 10]:
    rf_test = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42)
    rf_test.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, rf_test.predict(X_train))
    test_acc = accuracy_score(y_test, rf_test.predict(X_test))
    print(f"max_depth={depth} -> train: {train_acc:.2%}, test: {test_acc:.2%}")

max_depth=3 -> train: 55.94%, test: 56.02%
max_depth=5 -> train: 56.31%, test: 56.44%
max_depth=8 -> train: 56.88%, test: 56.50%
max_depth=10 -> train: 58.88%, test: 56.23%


In [21]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# Rebuild a clean Logistic Regression model to evaluate
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

cm = confusion_matrix(y_test, lr_pred, labels=['H', 'D', 'A'])

cm_df = pd.DataFrame(
    cm,
    index=[f'Actual {l}' for l in ['H', 'D', 'A']],
    columns=[f'Predicted {l}' for l in ['H', 'D', 'A']]
)
print(cm_df)

          Predicted H  Predicted D  Predicted A
Actual H         3647            0          538
Actual D         1365            0          591
Actual A         1237            0         1189


In [22]:
from sklearn.metrics import classification_report

print(classification_report(y_test, lr_pred, labels=['H', 'D', 'A']))

              precision    recall  f1-score   support

           H       0.58      0.87      0.70      4185
           D       0.00      0.00      0.00      1956
           A       0.51      0.49      0.50      2426

    accuracy                           0.56      8567
   macro avg       0.37      0.45      0.40      8567
weighted avg       0.43      0.56      0.48      8567



c:\Users\rohit\OneDrive\Documents\ML All in here\projects\FIFA World Cup Prediction model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\rohit\OneDrive\Documents\ML All in here\projects\FIFA World Cup Prediction model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\rohit\OneDrive\Documents\ML All in here\projects\FIFA World Cup Prediction model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and be

In [23]:
lr_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_balanced.fit(X_train, y_train)
lr_balanced_pred = lr_balanced.predict(X_test)

print(classification_report(y_test, lr_balanced_pred, labels=['H', 'D', 'A']))

              precision    recall  f1-score   support

           H       0.68      0.62      0.65      4185
           D       0.26      0.26      0.26      1956
           A       0.48      0.56      0.52      2426

    accuracy                           0.52      8567
   macro avg       0.48      0.48      0.48      8567
weighted avg       0.53      0.52      0.52      8567



In [24]:
rf_balanced = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight='balanced',
    random_state=42
)
rf_balanced.fit(X_train, y_train)
rf_pred = rf_balanced.predict(X_test)

print(classification_report(y_test, rf_pred, labels=['H', 'D', 'A']))

              precision    recall  f1-score   support

           H       0.69      0.57      0.63      4185
           D       0.25      0.29      0.27      1956
           A       0.48      0.57      0.52      2426

    accuracy                           0.51      8567
   macro avg       0.47      0.48      0.47      8567
weighted avg       0.53      0.51      0.51      8567



In [25]:
train_data = model_df[model_df['date'] < '2026-06-11']
val_data = model_df[model_df['date'] >= '2026-06-11']

X_train_final = train_data[feature_cols]
y_train_final = train_data['result']
X_val = val_data[feature_cols]
y_val = val_data['result']

model_final = LogisticRegression(max_iter=1000, class_weight='balanced')
model_final.fit(X_train_final, y_train_final)

val_pred = model_final.predict(X_val)

print("Training matches:", X_train_final.shape[0])
print("Validation matches (real World Cup 2026 games):", X_val.shape[0])
print()
print(classification_report(y_val, val_pred, labels=['H', 'D', 'A']))

Training matches: 42730
Validation matches (real World Cup 2026 games): 102

              precision    recall  f1-score   support

           H       0.74      0.58      0.65        48
           D       0.24      0.25      0.24        24
           A       0.56      0.73      0.64        30

    accuracy                           0.55       102
   macro avg       0.51      0.52      0.51       102
weighted avg       0.57      0.55      0.55       102



In [26]:
import os
import pickle

os.makedirs("../models", exist_ok=True)

with open("../models/match_predictor.pkl", "wb") as f:
    pickle.dump(model_final, f)

print("Model saved!")

Model saved!


In [27]:
def get_current_form(team, n=5):
    """A team's average points from their last N real matches (most recent form)."""
    hist = team_matches[team_matches['team'] == team].sort_values('date')
    if hist.empty:
        return 1.0  # neutral default if we have zero history for this team
    return hist['points'].tail(n).mean()

def get_match_features(team_a, team_b, neutral=True):
    """Translate two team names into the model's 4 expected feature columns."""
    home_rank = get_rank(team_a)
    away_rank = get_rank(team_b)
    features = {
        'is_home_advantage': 0 if neutral else 1,
        'home_team_form': get_current_form(team_a),
        'away_team_form': get_current_form(team_b),
        'rank_difference': away_rank - home_rank
    }
    return pd.DataFrame([features])

# Try it
X_new = get_match_features('Brazil', 'France')
print(X_new)

probs = model_final.predict_proba(X_new)
print(dict(zip(model_final.classes_, probs[0])))

   is_home_advantage  home_team_form  away_team_form  rank_difference
0                  0             2.0             2.4               -1
{'A': np.float64(0.39914351224283534), 'D': np.float64(0.3766190492077038), 'H': np.float64(0.22423743854946085)}


In [28]:
# Build one row per team: their current form + rank
all_teams = team_matches['team'].unique()

team_stats = pd.DataFrame({
    'team': all_teams,
    'current_form': [get_current_form(t) for t in all_teams],
    'rank': [get_rank(t) for t in all_teams]
})

team_stats.to_csv("../data/team_stats.csv", index=False)
print(team_stats.shape)
team_stats.head()

(337, 3)


,team,current_form,rank
0,Abkhazia,1.8,NaN
1,Afghanistan,1.0,150.0
2,Albania,0.0,66.0
3,Alderney,0.6,NaN
4,Algeria,1.4,31.0
